# Example 3
## Creating and editing CIM objects

Now that you know how to load a profile, this example shows how to **create object instances** from its classes, set their attributes, assign identifiers, work with physical units, and link objects together through CIM associations.

We start by importing the `CoreEquipment` custom profile from Example 2.

In [2]:
import cimgraph_demo.custom_profiles.CoreEquipment as cim

## Creating an object instance

Creating a CIM object is as simple as instantiating its dataclass. Every object is automatically assigned a unique identifier (a UUID) on creation.

- `pprint()` prints a formatted JSON-LD view of the object.
- The object's `__dict__` reveals all of its underlying fields, including the empty associations waiting to be populated.

In [2]:
new_line = cim.ACLineSegment()

In [3]:
new_line.pprint()

{
    "@id": "40d87815-e0ac-434b-9de1-4830133d3465",
    "@type": "ACLineSegment"
}


In [4]:
new_line.__dict__

{'identifier': UUID('40d87815-e0ac-434b-9de1-4830133d3465'),
 'mRID': '40d87815-e0ac-434b-9de1-4830133d3465',
 'description': None,
 'energyIdentCodeEic': None,
 'name': None,
 'shortName': None,
 'aggregate': None,
 'normallyInService': None,
 'EquipmentContainer': None,
 'OperationalLimitSet': [],
 'BaseVoltage': None,
 'Terminals': [],
 'length': None,
 'bch': None,
 'gch': None,
 'r': None,
 'x': None,
 'Clamp': [],
 'Cut': [],
 '__uuid__': <cimgraph.data_profile.identity.UUID_Meta at 0x7c2a55b22660>,
 '__json_ld__': '{"@id": "40d87815-e0ac-434b-9de1-4830133d3465", "@type": "ACLineSegment"}'}

## Identifiers and deterministic UUIDs

By default each object gets a random UUID. But identifiers can also be **deterministic**: given the same name and seed (domain URI), CIMantic Graphs generates the same UUID every time. This is what makes the two objects below compare as equal even though they were created independently.

You can also:
- Supply your own `mRID` (master resource identifier) while keeping a separate UUID.
- Call `uuid(name=..., seed=...)` to generate a reproducible identifier from a name and domain.

In [5]:
line1 = cim.ACLineSegment(name='my_line')
line2 = cim.ACLineSegment(name='my_line')
print(line1)
print(line2)
print(line1 == line2)

{"@id": "a7743a2d-626a-487d-b8e1-d6ebe778800f", "@type": "ACLineSegment", "name": "my_line"}
{"@id": "a7743a2d-626a-487d-b8e1-d6ebe778800f", "@type": "ACLineSegment", "name": "my_line"}
True


In [6]:
line3 = cim.ACLineSegment(mRID = 'local_id_12345')
line3.pprint(print_mRID=True)

{
    "@id": "4d12999f-de14-4573-8f29-b5e13fe4d562",
    "@type": "ACLineSegment",
    "identifier": "4d12999f-de14-4573-8f29-b5e13fe4d562",
    "mRID": "local_id_12345"
}


In [7]:
line4 = cim.ACLineSegment()
line4.uuid(name="my_line", seed='my_domain_uri')
line4.pprint()

{
    "@id": "14261c08-7625-4ddd-94f2-bd02623cc2a5",
    "@type": "ACLineSegment",
    "name": "my_line"
}


## Setting attribute values

Attributes can be set at construction time or afterward. For simple numeric attributes you can assign plain values directly:

In [8]:
line = cim.ACLineSegment(r = 0.005, x = 0.01)
line.pprint()

{
    "@id": "9b9a531f-1de3-4012-a763-5099d687aa59",
    "@type": "ACLineSegment",
    "r": "0.005",
    "x": "0.01"
}


### Working with physical units

Many CIM attributes are physical quantities. CIMantic Graphs wraps these in **`CIMUnit`** types (backed by `pint`) that store values internally in SI base units but accept any input unit you specify.

This means you never multiply or divide by `1e3`/`1e6` by hand — you pass the source unit to the constructor, and convert back out with `.to()`:

- `cim.Length(50, 'mile')` stores meters internally.
- `cim.Resistance('5', 'milliohm')` and `cim.Reactance(10, 'ohm', 'milli')` store ohms.

Notice in the output that `length` is stored as `80467.2` (meters) and `r` as `0.005` (ohms), regardless of the input units.

In [ ]:
line = cim.ACLineSegment()
line.length = cim.Length(50, 'mile')
line.r = cim.Resistance('5', 'milliohm')
line.x = cim.Reactance(10, 'ohm', 'milli')
line.pprint()

{
    "@id": "98becbc9-c301-45bf-a01f-872dd3330dee",
    "@type": "ACLineSegment",
    "length": "80467.2",
    "r": "0.005",
    "x": "0.01"
}


Reading a `CIMUnit` attribute back gives you its value in SI base units, and `.to()` converts it to any compatible unit on demand:

In [12]:
line.r

0.005 ohm

In [10]:
line.length.to('km')

80.4672

### Setting attributes with `setattr`

You can also set attributes dynamically with Python's built-in `setattr()`, which is handy when attribute names are determined at runtime (for example, when parsing data):

In [11]:
line = cim.ACLineSegment()
setattr(line,'r',0.005)
setattr(line,'x', 0.01)
line.pprint()

{
    "@id": "55d2d3f6-3f76-4ccb-aa58-c8d1655b580b",
    "@type": "ACLineSegment",
    "r": "0.005",
    "x": "0.01"
}


## Linking objects through associations

Real CIM models are graphs of interconnected objects. You build these connections by assigning one object to another's **association** attribute.

Here we construct a small hierarchy:

- a `BaseVoltage` of 115 kV,
- a `Substation` named `AIRPORT`,
- a `VoltageLevel` placed in that substation at the 115 kV base,
- and a `Breaker` contained in the voltage level.

Assigning `level_115.Substation = sub` and `breaker.EquipmentContainer = level_115` wires the equipment into its container hierarchy.

In [12]:
base_115kv = cim.BaseVoltage(name='base_115_kv')
base_115kv.nominalVoltage = cim.Voltage(115, 'kV')

sub = cim.Substation(name = 'AIRPORT')

level_115 = cim.VoltageLevel(name='airport_115kv')
level_115.Substation = sub
level_115.BaseVoltage = base_115kv

breaker = cim.Breaker(name='AIRP_115_5-1')
breaker.EquipmentContainer = level_115
breaker.BaseVoltage = base_115kv

breaker.pprint()

{
    "@id": "d31928e1-c224-48c1-b16b-96bdacf6497e",
    "@type": "Breaker",
    "name": "AIRP_115_5-1",
    "EquipmentContainer": {
        "@id": "41c03b51-e3b5-494d-8314-55304a60b9d4",
        "@type": "VoltageLevel"
    },
    "BaseVoltage": {
        "@id": "307ede82-5250-48ff-b1f4-90343264d35d",
        "@type": "BaseVoltage"
    }
}


### Reverse associations

CIM associations are bidirectional. Setting `breaker.EquipmentContainer` establishes the link in one direction; you can optionally populate the **reverse** side so the container also knows about its children. This lets you traverse the graph in either direction.

In [13]:
# Optionaly set reverse associations
sub.VoltageLevels.append(level_115)
level_115.Equipments.append(breaker)

### Traversing the graph

Once objects are linked, you can navigate the model by following association attributes — walking *up* from a breaker to its substation, or *down* from a substation through its voltage levels to the equipment they contain.

In [14]:
print(breaker.EquipmentContainer.Substation.name)

AIRPORT


In [15]:
for voltage_level in sub.VoltageLevels:
    for equipment in voltage_level.Equipments:
        print(equipment.name)

AIRP_115_5-1


## Summary

In this example you learned how to:

- Create CIM object instances and inspect them with `pprint()`.
- Assign identifiers, including custom `mRID`s and deterministic UUIDs.
- Set attributes directly, via `CIMUnit` types with explicit units, and with `setattr()`.
- Link objects through associations and traverse the resulting graph in both directions.

You now have the building blocks to assemble complete CIM network models.